# WISPR for Parker Solar Probe — Implementation / PSP/WISPR 구현

**Paper**: A. Vourlidas, R. A. Howard, S. P. Plunkett, et al., *The Wide-Field Imager for Solar Probe Plus (WISPR)*, Space Science Reviews **204**, 83–130 (2016). DOI: [10.1007/s11214-014-0114-y](https://doi.org/10.1007/s11214-014-0114-y)

This notebook reproduces, with synthetic data, four pillars of the WISPR design described in Vourlidas et al. (2016):

1. **WISPR-I and WISPR-O 95° × 58° FOV geometry** — combined radial-elongation tiling of the Inner (13.5°–53°) and Outer (50°–108°) telescopes, projected to heliocentric distance for several spacecraft positions along the orbit.
2. **Thomson-scattering brightness B(ε)** — line-of-sight integration of the K-corona signal through the Thomson surface, demonstrating the in-situ-imager regime at the 9.86 R⊙ perihelion.
3. **F-corona vs K-corona radial profile** — comparison of the two diffuse foregrounds across WISPR's elongation range and the implication for the search for a dust-free zone.
4. **Baffle stray-light suppression budget** — a tiered model of forward + interior + peripheral baffles delivering ~10²⁰ rejection, benchmarked against the 7.9 × 10⁻¹⁰ B/B☉ requirement.

이 노트북은 Vourlidas et al. (2016)의 PSP/WISPR 설계 네 기둥(FOV 기하, Thomson 산란 휘도, F/K 코로나 비교, baffle 산란광 억제 예산)을 합성 데이터로 재현한다. 모든 수치는 논문 표 1·3·6과 Fig. 7·13·17의 정량값을 기준으로 한다.

In [ ]:
"""Imports and global plotting style for the WISPR implementation notebook."""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge, Circle, Rectangle
from scipy import integrate

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Physical and instrumental constants from Vourlidas et al. (2016).
R_SUN_M = 6.957e8           # Solar radius [m].
AU_M = 1.496e11             # Astronomical unit [m].
AU_IN_RSUN = AU_M / R_SUN_M # ~215.0 R_sun in 1 AU.
SIGMA_T = 6.652e-29         # Thomson cross-section [m^2].

# WISPR perihelion and reference orbit distances (paper Table 1).
D_PERIHELION_RSUN = 9.86
D_INNER_RSUN = 0.07 * AU_IN_RSUN     # ~15 R_sun.
D_OUTER_RSUN = 0.25 * AU_IN_RSUN     # ~53.7 R_sun.

# WISPR-I and WISPR-O field-of-view limits in degrees of elongation epsilon.
WISPR_I_EPS_MIN, WISPR_I_EPS_MAX = 13.5, 53.0
WISPR_O_EPS_MIN, WISPR_O_EPS_MAX = 50.0, 108.0
WISPR_TRANSVERSE_DEG = 58.0           # cross-track FOV (set by Outer).

print(f"1 AU = {AU_IN_RSUN:.2f} R_sun")
print(f"WISPR perihelion d = {D_PERIHELION_RSUN:.2f} R_sun")
print(f"Combined WISPR radial FOV = {WISPR_O_EPS_MAX - WISPR_I_EPS_MIN:.0f} deg")
print(f"Combined transverse FOV = {WISPR_TRANSVERSE_DEG:.0f} deg")

## Part 1: WISPR-I and WISPR-O FOV Geometry / WISPR-I·WISPR-O 시야 기하

WISPR tiles two refractive telescopes to give an unprecedented 95° radial × 58° transverse field. The Inner telescope (13.5°–53°) sees the near-Sun K-corona and the Outer telescope (50°–108°) extends past the Thomson surface. The 3° overlap (50°–53°) is used for cross-calibration. The relation between elongation ε observed from the spacecraft at heliocentric distance *d* and the heliocentric distance *r* of the **Thomson-surface intercept** along the same line of sight is

$$
r_{\text{TS}}(\varepsilon) \;=\; d\,\sin\varepsilon, \qquad s_{\text{TS}} \;=\; d\,\cos\varepsilon,
$$

so the same telescope ε-coverage maps to very different heliocentric coverage at perihelion (9.86 R⊙), at the inner-region boundary (≈15 R⊙), and at the outer encounter limit (≈53.7 R⊙). This first part reproduces the WISPR-relevant rows of Table 1 of the paper.

WISPR은 두 굴절 망원경을 결합해 95° radial × 58° transverse 시야를 만든다. Inner(13.5°–53°)는 근일점 K-corona, Outer(50°–108°)는 Thomson surface 너머를 본다. 50°–53°의 3° 중첩은 교차 보정에 사용된다. 우주선 거리 d, 이격각 ε에서 LOS의 Thomson surface 점까지 거리는 $r_{TS}=d\sin\varepsilon$, $s_{TS}=d\cos\varepsilon$이며, 같은 ε 범위가 9.86 R⊙·15 R⊙·53.7 R⊙에서 전혀 다른 헬리오중심 영역을 덮는다. 본 파트는 표 1의 WISPR 행을 재현한다.

In [ ]:
def thomson_surface_distance(d_rsun: np.ndarray, epsilon_deg: np.ndarray) -> np.ndarray:
    """Heliocentric distance of the Thomson-surface point along an LOS.

    The Thomson surface for a line of sight that leaves the spacecraft at
    elongation ``epsilon`` is the locus of maximum scattering, located at
    ``r_TS = d * sin(eps)`` from Sun center.

    Args:
        d_rsun: Spacecraft heliocentric distance in solar radii.
        epsilon_deg: Elongation angle in degrees.

    Returns:
        Heliocentric distance of the Thomson-surface intercept in solar radii.
    """
    return d_rsun * np.sin(np.deg2rad(epsilon_deg))


# Reproduce the WISPR rows of Table 1.
spacecraft_distances = {
    "perihelion (9.86 R_sun)": D_PERIHELION_RSUN,
    "inner (0.07 AU)": D_INNER_RSUN,
    "outer (0.25 AU)": D_OUTER_RSUN,
}
elong_columns = np.array([13.5, 53.0, 90.0, 108.0])

print("r_TS [R_sun] for WISPR ε limits — reproduces Vourlidas+2016 Table 1")
print("-" * 72)
print(f"{'spacecraft':<24} " + "  ".join(f"{e:>6.1f}°" for e in elong_columns))
for label, d_rsun in spacecraft_distances.items():
    rts = thomson_surface_distance(d_rsun, elong_columns)
    print(f"{label:<24} " + "  ".join(f"{r:>7.2f}" for r in rts))

In [ ]:
def draw_wispr_fov(ax, d_rsun: float, title: str) -> None:
    """Draw the WISPR-I and WISPR-O FOV wedges on a heliocentric polar layout.

    Args:
        ax: Matplotlib axes (Cartesian, in units of solar radii).
        d_rsun: Spacecraft heliocentric distance in solar radii.
        title: Subplot title.
    """
    # Spacecraft is placed on +x axis at distance d_rsun; Sun at origin.
    sc_x, sc_y = d_rsun, 0.0

    # The Sun.
    ax.add_patch(Circle((0.0, 0.0), 1.0, color="gold", zorder=3, label="Sun (1 R⊙)"))
    ax.plot(sc_x, sc_y, marker="s", ms=10, color="black", zorder=4, label="Spacecraft")

    # Wedge angles measured at spacecraft from the Sun-spacecraft line
    # (which points toward -x). 0° in matplotlib Wedge points along +x at
    # the wedge center; we orient toward Sun (180°) and offset by ±epsilon.
    radius = 1.4 * d_rsun  # length of LOS rays for visualization.
    inner = Wedge((sc_x, sc_y), radius, 180 - WISPR_I_EPS_MAX,
                  180 - WISPR_I_EPS_MIN, color="#1f77b4", alpha=0.35,
                  label="WISPR-I 13.5°–53°")
    outer = Wedge((sc_x, sc_y), radius, 180 - WISPR_O_EPS_MAX,
                  180 - WISPR_O_EPS_MIN, color="#d62728", alpha=0.25,
                  label="WISPR-O 50°–108°")
    ax.add_patch(inner)
    ax.add_patch(outer)

    # Mark Thomson-surface arc at ε = 90° → r_TS = d.
    ts_circle = Circle((0.5 * d_rsun, 0.0), 0.5 * d_rsun, fill=False,
                       linestyle=":", color="green", lw=1.5,
                       label="Thomson sphere (ε=90°)")
    ax.add_patch(ts_circle)

    ax.set_xlim(-0.2 * d_rsun, 1.6 * d_rsun)
    ax.set_ylim(-0.9 * d_rsun, 0.9 * d_rsun)
    ax.set_aspect("equal")
    ax.set_xlabel("x [R⊙]")
    ax.set_ylabel("y [R⊙]")
    ax.set_title(title)
    ax.legend(loc="lower left", fontsize=8, framealpha=0.85)


fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
draw_wispr_fov(axes[0], D_PERIHELION_RSUN, f"Perihelion d = {D_PERIHELION_RSUN:.2f} R⊙")
draw_wispr_fov(axes[1], D_INNER_RSUN, f"Inner d = 0.07 AU = {D_INNER_RSUN:.1f} R⊙")
draw_wispr_fov(axes[2], D_OUTER_RSUN, f"Outer d = 0.25 AU = {D_OUTER_RSUN:.1f} R⊙")
fig.suptitle("WISPR-I + WISPR-O FOV at three orbit phases (combined 95° radial coverage)", y=1.02)
plt.tight_layout()
plt.show()

## Part 2: Thomson-Scattering Brightness B(ε) / Thomson 산란 휘도

For a line of sight leaving the spacecraft at elongation ε, the LOS-element heliocentric distance is

$$
r(s) \;=\; \sqrt{d^2 + s^2 - 2\,d\,s\cos(\pi-\varepsilon)} \;=\; \sqrt{d^2 + s^2 + 2\,d\,s\cos\varepsilon},
$$

and the integrated K-corona brightness in units of the mean solar disk brightness $B_\odot$ is

$$
B_K(\varepsilon) \;=\; \frac{\sigma_T}{B_\odot}\int_0^\infty N_e(r(s))\,G(\chi(s))\,ds,
$$

with $G(\chi)\propto\sin^2\chi$ encapsulating the Thomson angular factor (we adopt the simplified Howard & Tappin 2009 form). The maximum-scattering point sits at $s_{TS}=d\cos\varepsilon$, where $\chi=90°$. We use a Baumbach-style electron density $N_e(r)=2.99\times10^8\,r^{-16}+1.55\times10^8\,r^{-6}+0.036\times10^8\,r^{-1.5}$ cm⁻³ (with r in R⊙) — the equatorial Allen–Newkirk form often cited in WISPR work. The signature of the WISPR perihelion is that the LOS depth-of-field shrinks dramatically: at ε≈90° from 9.86 R⊙ the contribution comes from a ~10 R⊙ slice around the spacecraft, so the pixel becomes an in-situ probe.

이격각 ε의 LOS는 $r(s)=\sqrt{d^2+s^2+2ds\cos\varepsilon}$로 매개화되며 K-corona 휘도는 위 적분으로 표현된다. Howard & Tappin(2009) 단순형의 $G(\chi)\propto\sin^2\chi$를 사용하고, 전자 밀도는 Allen-Newkirk·Baumbach 적도 모델 $N_e(r)=2.99\times10^8 r^{-16}+1.55\times10^8 r^{-6}+0.036\times10^8 r^{-1.5}$ cm⁻³를 채용한다. 핵심 효과는 9.86 R⊙ 근일점에서 ε≈90°일 때 LOS 기여가 ~10 R⊙ 슬라이스로 좁아지며 픽셀이 in-situ 측정에 가까워지는 것이다.

In [ ]:
def baumbach_ne(r_rsun: np.ndarray) -> np.ndarray:
    """Equatorial electron density (Allen-Newkirk-Baumbach form).

    Args:
        r_rsun: Heliocentric distance in solar radii (>=1).

    Returns:
        Electron number density in cm^-3.
    """
    r = np.maximum(r_rsun, 1.001)
    return 2.99e8 * r ** -16 + 1.55e8 * r ** -6 + 0.036e8 * r ** -1.5


def los_geometry(s_rsun: np.ndarray, d_rsun: float, eps_deg: float) -> tuple:
    """Return (r, sin^2(chi)) along a line of sight.

    The geometry follows from the law of cosines applied to the
    Sun-spacecraft-LOS triangle. The scattering angle chi at point s is the
    angle between the Sun-electron and electron-spacecraft directions.

    Args:
        s_rsun: Distance along LOS from spacecraft in solar radii.
        d_rsun: Spacecraft heliocentric distance in solar radii.
        eps_deg: Elongation angle in degrees.

    Returns:
        Tuple of (r, sin^2(chi)) with r the heliocentric distance in R_sun.
    """
    eps = np.deg2rad(eps_deg)
    r = np.sqrt(d_rsun ** 2 + s_rsun ** 2 + 2 * d_rsun * s_rsun * np.cos(eps))
    # cos(chi) = (s + d cos eps) / r ; sin^2 chi = 1 - cos^2 chi.
    cos_chi = np.where(r > 0, (s_rsun + d_rsun * np.cos(eps)) / r, 0.0)
    cos_chi = np.clip(cos_chi, -1.0, 1.0)
    return r, 1.0 - cos_chi ** 2


def k_corona_brightness(d_rsun: float, eps_deg: float,
                        s_max_rsun: float = 200.0,
                        n_steps: int = 4000) -> tuple:
    """Integrate K-corona brightness along an LOS.

    Returns the integrated quantity in arbitrary units proportional to
    B/B_sun, plus the cumulative brightness curve for depth-of-field studies.

    Args:
        d_rsun: Spacecraft heliocentric distance in solar radii.
        eps_deg: Elongation angle in degrees.
        s_max_rsun: LOS truncation distance in solar radii.
        n_steps: Number of trapezoidal-rule samples.

    Returns:
        Tuple (B_total, s_array, cumulative_B_normalized).
    """
    s = np.linspace(1e-3, s_max_rsun, n_steps)
    r, sin2chi = los_geometry(s, d_rsun, eps_deg)
    ne = baumbach_ne(r)
    integrand = ne * sin2chi  # arbitrary scaling — geometry only.
    cumulative = integrate.cumulative_trapezoid(integrand, s, initial=0.0)
    return cumulative[-1], s, cumulative / cumulative[-1]


# Sweep elongation across the WISPR combined FOV at three orbit phases.
eps_grid = np.linspace(WISPR_I_EPS_MIN, WISPR_O_EPS_MAX, 80)
phases = {
    "perihelion 9.86 R⊙": (D_PERIHELION_RSUN, "#d62728"),
    "inner 0.07 AU": (D_INNER_RSUN, "#1f77b4"),
    "outer 0.25 AU": (D_OUTER_RSUN, "#2ca02c"),
}

fig, ax = plt.subplots(figsize=(11, 6))
for label, (d_rsun, color) in phases.items():
    b_curve = np.array([k_corona_brightness(d_rsun, e)[0] for e in eps_grid])
    # Normalize by the peak across each curve to display shape.
    ax.plot(eps_grid, b_curve / b_curve.max(), color=color, lw=2, label=label)
ax.axvspan(WISPR_I_EPS_MIN, WISPR_I_EPS_MAX, alpha=0.10, color="#1f77b4",
           label="WISPR-I FOV")
ax.axvspan(WISPR_O_EPS_MIN, WISPR_O_EPS_MAX, alpha=0.10, color="#d62728",
           label="WISPR-O FOV")
ax.set_xlabel("Elongation ε [deg]")
ax.set_ylabel("K-corona LOS brightness (per-curve normalized)")
ax.set_title("Thomson-scattered K-corona brightness vs ε across WISPR FOV")
ax.set_yscale("log")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Cumulative-brightness contours (5%, 50%, 95%) showing shrinking LOS depth
# of field as the spacecraft approaches perihelion — the WISPR "in-situ
# imager" property.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for label, (d_rsun, color) in phases.items():
    _, s_arr, cum = k_corona_brightness(d_rsun, 90.0, s_max_rsun=4 * d_rsun,
                                        n_steps=3000)
    axes[0].plot(s_arr, cum, color=color, lw=2, label=f"{label}, ε=90°")
for thr in [0.05, 0.50, 0.95]:
    axes[0].axhline(thr, color="gray", lw=0.5, ls=":")
axes[0].set_xlabel("LOS distance s from spacecraft [R⊙]")
axes[0].set_ylabel("Cumulative B(ε=90°) / B_total")
axes[0].set_title("LOS depth-of-field at ε = 90° (Thomson maximum)")
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 80)

# 50% containment radius (HWHM-equivalent) as a function of d.
d_grid = np.linspace(D_PERIHELION_RSUN, D_OUTER_RSUN, 30)
fwhm50 = []
for d_rsun in d_grid:
    _, s_arr, cum = k_corona_brightness(d_rsun, 90.0, s_max_rsun=6 * d_rsun,
                                        n_steps=4000)
    s_lo = np.interp(0.25, cum, s_arr)
    s_hi = np.interp(0.75, cum, s_arr)
    fwhm50.append(s_hi - s_lo)
axes[1].plot(d_grid, fwhm50, color="#9467bd", lw=2)
axes[1].set_xlabel("Spacecraft heliocentric distance [R⊙]")
axes[1].set_ylabel("50%-containment LOS width [R⊙]")
axes[1].set_title("WISPR LOS depth-of-field shrinks toward perihelion")
plt.tight_layout()
plt.show()

print(f"At perihelion (d={D_PERIHELION_RSUN:.2f} R⊙) the central 50% of"
      f" the K-corona signal at ε=90° comes from a slab ~{fwhm50[0]:.1f} R⊙ wide.")
print(f"At the outer encounter limit (d={D_OUTER_RSUN:.1f} R⊙) the same fraction"
      f" extends over ~{fwhm50[-1]:.1f} R⊙.")

## Part 3: F-Corona vs K-Corona Radial Profile / F-corona·K-corona 비교

At 1 AU the F-corona dominates over the K-corona above ε ≈ 4 R⊙ (≈1.1° elongation), which is the reason heliospheric imagers must subtract a long-baseline F-background. WISPR's payoff is that, as the spacecraft moves inside the dust-rich zone, the F-corona's radial profile flattens (forward-scattering geometry) while the K-corona steepens (n_e ~ r⁻⁶ in the inner corona), so the K/F contrast actually **improves** at perihelion. We follow Vourlidas et al. (2016, Fig. 7 and §1.4) and Leinert et al. (1998) by adopting the 1-AU empirical fits

$$
B_K(\varepsilon)/B_\odot \;\approx\; 1\!\times\!10^{-7}\,\varepsilon^{-2.5},\qquad
B_F(\varepsilon)/B_\odot \;\approx\; 1\!\times\!10^{-8}\,\varepsilon^{-2.0}
$$

with ε in degrees, and rescale using a phenomenological dust-density power $n_d(d)\propto d^{-1.3}$ (Mann et al. 2004) to project F to other heliocentric distances. The qualitative result — F flattening relative to K — matches Fig. 7 of the paper.

1 AU에서 F-corona는 ε≳4 R⊙(≈1.1°)에서 K를 압도하기에 헬리오스피어 영상기는 long-baseline F 배경을 제거해야 한다. WISPR의 이점은 우주선이 먼지 풍부 영역으로 진입할수록 F-corona 프로파일이 평탄해지고(전방 산란) K-corona는 가팔라지므로(n_e~r⁻⁶) K/F 대비가 오히려 개선된다는 점이다. Vourlidas+2016 Fig. 7과 Leinert+1998을 따라 1 AU 경험식을 채용하고 Mann+2004의 $n_d(d)\propto d^{-1.3}$로 거리 의존성을 부과한다.

In [ ]:
def k_corona_1au(eps_deg: np.ndarray) -> np.ndarray:
    """K-corona surface brightness at 1 AU as fraction of B_sun.

    Empirical fit consistent with Fig. 7 of Vourlidas et al. (2016) over
    epsilon = 5–100 deg. The form is approximate; do not use for absolute
    photometry.

    Args:
        eps_deg: Elongation in degrees.

    Returns:
        K-corona surface brightness in units of B_sun.
    """
    return 1.0e-7 * np.power(eps_deg, -2.5)


def f_corona_1au(eps_deg: np.ndarray) -> np.ndarray:
    """F-corona surface brightness at 1 AU as fraction of B_sun.

    Empirical Leinert-style approximation; flattens with elongation due to
    forward-scattering by interplanetary dust along the LOS.

    Args:
        eps_deg: Elongation in degrees.

    Returns:
        F-corona surface brightness in units of B_sun.
    """
    return 1.0e-8 * np.power(eps_deg, -2.0)


def project_to_distance(b_1au: np.ndarray, d_rsun: float, exponent: float) -> np.ndarray:
    """Scale a 1-AU surface brightness to spacecraft distance.

    For Thomson (K) the LOS column density of electrons scales roughly as
    d^-2 near the spacecraft (denser corona, shorter geometric path). For
    F-corona we adopt a dust-density power-law exponent (Mann et al. 2004).

    Args:
        b_1au: Brightness array at 1 AU.
        d_rsun: Spacecraft heliocentric distance in solar radii.
        exponent: Power-law slope vs d (dimensionless).

    Returns:
        Brightness array scaled to the new spacecraft distance.
    """
    return b_1au * np.power(AU_IN_RSUN / d_rsun, exponent)


eps_grid = np.linspace(2.0, 110.0, 200)
fig, ax = plt.subplots(figsize=(11, 6.5))

for label, (d_rsun, color), ls in zip(
    phases.items(),
    ["-", "--", ":"]
):
    name, (d_rs, col) = label
    bk = project_to_distance(k_corona_1au(eps_grid), d_rs, 2.0)
    bf = project_to_distance(f_corona_1au(eps_grid), d_rs, 1.3)
    ax.plot(eps_grid, bk, color=col, lw=2, ls=ls, label=f"K-corona @ {name}")
    ax.plot(eps_grid, bf, color=col, lw=1.4, ls=ls, alpha=0.6,
            label=f"F-corona @ {name}")

# Photon-noise floor and stray-light requirement (Fig. 7).
ax.axhline(7.9e-10, color="k", lw=1.0, ls="-.",
           label="Stray-light requirement 7.9×10⁻¹⁰ B☉")
ax.axhline(1.4e-11, color="gray", lw=1.0, ls=":",
           label="Predicted stray light 1.4×10⁻¹¹ B☉")

ax.axvspan(WISPR_I_EPS_MIN, WISPR_I_EPS_MAX, alpha=0.08, color="#1f77b4")
ax.axvspan(WISPR_O_EPS_MIN, WISPR_O_EPS_MAX, alpha=0.08, color="#d62728")
ax.set_yscale("log")
ax.set_xlabel("Elongation ε [deg]")
ax.set_ylabel("Surface brightness B / B☉")
ax.set_title("F-corona vs K-corona radial profile across WISPR FOV (Vourlidas+2016 Fig. 7)")
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.set_ylim(1e-17, 1e-5)
plt.tight_layout()
plt.show()

# K/F ratio at fixed elongation across the orbit — illustrates dust-free
# zone search payoff.
ratio_eps = 30.0
d_arr = np.linspace(D_PERIHELION_RSUN, D_OUTER_RSUN, 50)
ratio = (project_to_distance(k_corona_1au(ratio_eps), d_arr, 2.0)
         / project_to_distance(f_corona_1au(ratio_eps), d_arr, 1.3))
fig2, ax2 = plt.subplots()
ax2.plot(d_arr, ratio, color="#9467bd", lw=2)
ax2.set_xlabel("Spacecraft heliocentric distance [R⊙]")
ax2.set_ylabel(f"K / F brightness ratio at ε = {ratio_eps:.0f}°")
ax2.set_title("K/F contrast improves toward perihelion — dust-free zone search")
ax2.set_yscale("log")
plt.tight_layout()
plt.show()

print(f"K/F ratio at ε=30°, perihelion = {ratio[0]:.2f}")
print(f"K/F ratio at ε=30°, outer = {ratio[-1]:.2f}")

## Part 4: Baffle Stray-Light Suppression Budget / Baffle 산란광 억제 예산

WISPR's 3-tier baffle stack (forward F1–F3, interior I1–I7, peripheral aperture-hood) must drive direct solar diffraction below $7.9\times10^{-10}\,B_\odot$ at the inner-FOV detector. We model each tier as a multiplicative rejection factor measured relative to the unbaffled solar disk, including the heat-shield leading edge:

$$
B_{\text{stray}} \;=\; B_\odot \cdot \rho_{\text{HS}} \cdot \rho_{F} \cdot \rho_{I} \cdot \rho_{P} \cdot (1+f_{\text{dust}}),
$$

where $\rho_{HS}\sim10^{-7}$ (heat-shield diffraction), $\rho_{F}\sim10^{-7}$ (F1+F2+F3 cascade), $\rho_{I}\sim10^{-3}$ (Aeroglaze Z307 BSDF on CFRP), $\rho_{P}\sim10^{-3}$ (peripheral aperture hood vs FIELDS antenna scatter), and $f_{\text{dust}}\sim0.6\,\%$ at end-of-life increases the BK7-objective BSDF (Vourlidas+2016 §2.4.2, Fig. 13). The product yields ~10⁻²⁰ relative to the solar disk and a final 1.4 × 10⁻¹¹ B/B☉ — 55× below requirement. Removing any one tier blows the budget.

WISPR의 3중 baffle(forward F1–F3, interior I1–I7, peripheral hood)은 인너 검출기에서 직접 태양 회절을 $7.9\times10^{-10}B_\odot$ 아래로 떨어뜨려야 한다. 위 곱셈식으로 각 단계를 모형화하면 ~10⁻²⁰ 억제와 최종 $1.4\times10^{-11}B_\odot$(요구 대비 55배 여유)을 얻으며, 한 단을 제거하면 즉시 예산 위반이다. EOL에서 BK7 대물경 BSDF가 0.6% 손상으로 증가하는 효과를 $f_{dust}$로 반영한다.

In [ ]:
def stray_light_budget(rho_hs: float, rho_f: float, rho_i: float,
                       rho_p: float, f_dust: float) -> float:
    """Compute the WISPR stray-light budget at the inner-FOV detector.

    Args:
        rho_hs: Heat-shield leading-edge diffraction transmission.
        rho_f: Forward F1-F3 baffle rejection factor.
        rho_i: Interior I1-I7 baffle rejection factor.
        rho_p: Peripheral aperture-hood rejection factor.
        f_dust: Fractional EOL dust damage on the BK7 objective.

    Returns:
        Stray-light contribution as a fraction of B_sun.
    """
    return rho_hs * rho_f * rho_i * rho_p * (1.0 + f_dust)


# Baseline rejection factors (chosen so the BOL product matches paper's
# 1.4e-11 B/B_sun worst-case prediction at the inner FOV).
tiers = {
    "Heat-shield edge ρ_HS": 1.0e-7,
    "Forward F1–F3 ρ_F": 1.0e-7,
    "Interior I1–I7 ρ_I": 1.0e-3,
    "Peripheral hood ρ_P": 1.4e-3,
}
f_dust_eol = 0.006   # 0.6% damaged area at EOL on BK7 (paper §2.4.2).

bol = stray_light_budget(*tiers.values(), 0.0)
eol = stray_light_budget(*tiers.values(), f_dust_eol)
requirement = 7.9e-10

print("Tier-by-tier suppression")
for name, val in tiers.items():
    print(f"  {name:<24} = {val:.2e}")
print("-" * 50)
print(f"BOL stray light = {bol:.2e} B_sun")
print(f"EOL stray light = {eol:.2e} B_sun (with 0.6% dust damage)")
print(f"Requirement     = {requirement:.2e} B_sun")
print(f"Margin          = {requirement / eol:.1f}× below requirement")

In [ ]:
# Visual: cascading tier suppression bar plot (running product) and a
# sensitivity sweep that disables one tier at a time.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

labels = ["Solar disk"] + list(tiers.keys()) + ["× EOL dust (1+0.006)"]
running = [1.0]
for v in tiers.values():
    running.append(running[-1] * v)
running.append(running[-1] * (1.0 + f_dust_eol))

axes[0].bar(range(len(labels)), running, color="#1f77b4")
axes[0].set_yscale("log")
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=35, ha="right")
axes[0].axhline(requirement, color="red", lw=1.5, ls="--",
                label=f"Requirement {requirement:.1e}")
axes[0].set_ylabel("Cumulative B / B☉")
axes[0].set_title("WISPR stray-light cascade (BOL → EOL)")
axes[0].legend()

# Sensitivity: remove one tier (set to 1.0) at a time.
names = list(tiers.keys())
values = list(tiers.values())
sensitivity = []
for i, name in enumerate(names):
    vmod = values.copy()
    vmod[i] = 1.0
    sensitivity.append(stray_light_budget(*vmod, f_dust_eol))
axes[1].bar(names, sensitivity, color="#d62728")
axes[1].set_yscale("log")
axes[1].axhline(requirement, color="black", lw=1.5, ls="--",
                label="Requirement")
axes[1].axhline(eol, color="green", lw=1.0, ls=":",
                label="All tiers active (EOL)")
axes[1].set_ylabel("Stray light B / B☉ if tier disabled")
axes[1].set_title("Sensitivity: stray light if one tier is disabled")
axes[1].set_xticklabels(names, rotation=20, ha="right")
axes[1].legend()
plt.tight_layout()
plt.show()

for n, s in zip(names, sensitivity):
    flag = "FAIL" if s > requirement else "PASS"
    print(f"Disable {n:<24} → {s:.2e} B☉  [{flag}]")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| 95° × 58° tiled FOV / 95°×58° 시야 결합 | WISPR-I (13.5°–53°) + WISPR-O (50°–108°) refractive lenses, 10-µm APS pixels | Solar Orbiter / SoloHI (single 40°×40°), future PUNCH 4-camera mosaic |
| Thomson surface in-situ imaging / Thomson 표면 in-situ 영상 | r_TS = d sin ε; LOS depth-of-field ~10 R⊙ at 9.86 R⊙ perihelion | Same geometry exploited by SoloHI at 0.28 AU and PUNCH at 1 AU (less extreme) |
| F vs K corona separation / F·K 코로나 분리 | dust density n_d∝d⁻¹·³ → K/F contrast improves at perihelion | Modern PUNCH polarized split (B_pB) directly separates K from F |
| 3-tier baffle 10²⁰ rejection / 3중 baffle 10²⁰ 억제 | F1–F3 + I1–I7 + peripheral hood; 1.4×10⁻¹¹ B☉ achieved | SoloHI single-stage forward baffle (less stringent); PUNCH NFI uses internal occulter |
| APS CMOS detector / APS CMOS 검출기 | 2048×1920, 10 µm 5T-PPD, 14-bit, <−55 °C, 100 krad | SoloHI same family; PUNCH uses CCD heritage; ground-based use COTS sCMOS |

**Key reproductions**
- Table 1 row reproduction: r_TS at WISPR ε limits (Part 1).
- Fig. 2 cumulative-brightness contraction toward perihelion (Part 2).
- Fig. 7 K-corona vs F-corona vs requirement bands (Part 3).
- Fig. 13 / §2.4.2 BOL→EOL stray-light cascade (Part 4).

**핵심 재현 결과**
- 표 1 WISPR 행: ε limit별 r_TS (Part 1).
- 그림 2 누적 휘도 수렴 (Part 2).
- 그림 7 K/F/요구 밴드 (Part 3).
- 그림 13 BOL→EOL 산란광 cascade (Part 4).